In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.VICI_valves.dim import DIM
from Instruments.VICI_valves.fsm import FSM
from Instruments.Syr_pumps.HB_syr_pump import HBElite
from Instruments.Knauer.knauer_pump_azura import KnauerPumpAzura
from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation
from Core.experiment_context import ExperimentManager

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump (Knauer) ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("Segmentation_testing")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=155.5,
    y_offset=10,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

In [4]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, and DIM
rsg = RSG(gilson=g, pump=syr_pump, dim=dim)

In [5]:
# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [ ]:
# --- ExperimentManager + RSG slug test ---
manager = ExperimentManager()
manager.load_experiment("test_001")
manager.begin_run()

slug = manager.get_next_slug()
result = rsg.create_slug(slug)

print(slug)
print(result)

In [97]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [9]:
k_pump.set_flow_rate(1000)
k_pump.start_flow()

[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started


'ON:OK'

In [91]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [39]:
rsg.pickup_from_vial("rack2", 1, 80, 0.5)

[Pickup] 80uL from rack2 vial 1 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [40]:
rsg.dispense_in_dim(90)

[DIM] 90uL dispensed in DIM @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [87]:
rsg.pickup_from_vial("rack2", 1, 80, 0.5)
rsg.dispense_in_vial("rack2", 2, 80, 0.5)

[Pickup] 80uL from rack2 vial 1 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Dispense] 80uL in rack2 vial 2 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [90]:
rsg.take_air_gap(20, 0.1)

[Air Gap] 20uL @ 0.1mL/min


In [92]:
fsm.gas_to_dim()

[FSM] Valve moved to A -> state = GAS_TO_DIM


In [93]:
rsg.pickup_from_vial("rack1", 3, 100, 0.25)

[Pickup] 100uL from rack1 vial 3 @ 0.25mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [94]:
dim.load()

[DIM] Valve moved to B -> state = DIMState.LOAD


In [95]:
rsg.dispense_in_dim(100)

[DIM] 100uL dispensed in DIM @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


In [89]:
syr_pump.infuse_volume(10, 0.25)

In [96]:
dim.inject()
fsm.carrier_to_dim()

[DIM] Valve moved to A -> state = DIMState.INJECT
[FSM] Valve moved to B -> state = CARRIER_TO_DIM


In [97]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0), 20)

In [98]:
k_pump.set_flow_rate(1000)
k_pump.start_flow()

[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started


'ON:OK'

In [99]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [6]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [48]:
# Override the defaults to target the System layer (-1)
result = g.send_command(
    command_name="Get NVM",
    device="System",
    device_id=-1,
    parameters={"UnitId": 35, "Address": 5}
)

print(result)


Get NVM: 2, 5=175


In [40]:
result = g.send_command(
    command_name="Set NVM",
    device="System",
    device_id=-1,
    parameters={"UnitId": 35, "Address": 27, "Value": "0.000"}
)


In [14]:
g.send_command("Get Z Position")

'Get Z Position: 175'

In [10]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [19]:
print(runze.state)
print(dim.state)

RunzeState.GAS_TO_DIM
DIMState.INJECT


In [7]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [6]:
rsg.take_air_gap(15, 0.05)

[Air Gap] 15uL @ 0.05mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [9]:
runze.gas_to_dim()

[Runze] Valve at pos 2 -> state = CARRIER_TO_DIM
[Runze] valve moved to pos 1 -> state = GAS_TO_DIM


In [10]:
dim.load()

[DIM] Valve moved to B -> state = DIMState.LOAD


In [11]:
rsg.pickup_from_vial("rack1", 1, 100, 0.2)

[Pickup] 100uL from rack1 vial 1 @ 0.2mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [29]:
rsg.dispense_in_vial("rack2", 2, 150, 0.5)

[Dispense] 150uL in rack2 vial 2 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [12]:
g.go_into_dim()

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(9.0, 104.0, 79)

In [16]:
syr_pump.infuse_volume(20, 0.4)

In [17]:
dim.inject()

[DIM] Valve moved to A -> state = DIMState.INJECT


In [20]:
runze.carrier_to_dim()

[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM


In [18]:
rsg.pickup_from_vial("rack2", 1, 50, 0.2)

[Pickup] 50uL from rack2 vial 1 @ 0.2mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


In [18]:
rsg.dispense_in_vial("rack2", 2, 200, 0.5)

[Dispense] 200uL in rack2 vial 2 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


In [26]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(155.5), np.float64(8.0))

In [40]:
rsg.dispense_in_vial("rack2", 2, 175, 0.5)

[Dispense] 175uL in rack2 vial 2 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


In [21]:
runze.carrier_to_dim()

[Runze] Valve at pos 1 -> state = GAS_TO_DIM
[Runze] valve moved to pos 2 -> state = CARRIER_TO_DIM


In [22]:
k_pump.set_flow_rate(1000)
k_pump.start_flow()

[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started


'ON:OK'

In [23]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [33]:
g.go_into_dim()

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(9.0, 104.0, 79)

In [35]:
syr_pump.infuse_volume(50, 0.4)

In [9]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127


(np.float64(319.0), np.float64(106.0), 20)

In [17]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(39.0))

In [16]:
g.move_z(119)

'Moved Z to 119 at 125 mm/sec. Result: Move Z with Speed: 2, Success'

In [20]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(39.0))

In [19]:
g.move_z(119)

'Moved Z to 119 at 125 mm/sec. Result: Move Z with Speed: 2, Success'

In [20]:
print(g.current_x)
print(g.current_y)

156.0
8.0


In [18]:
def run_validation(
    rsg,
    analyte_volumes,
    start_vial_pos=2,
    n_replicates=2,
    prepickup_volume=20,   # µL
    airgap1_volume=20,     # µL
    airgap2_volume=5,      # µL
    withdraw_rate=0.2,     # mL/min
    infuse_rate=0.5,       # mL/min
    #withdraw_delay=2.0     # seconds (NEW)
):
    """
    Automated syringe pump validation workflow.
    """
    
    print("=== Starting syringe pump validation run ===")
    
    vial_pos = start_vial_pos
    half_airgap1 = airgap1_volume / 2

    for vol in analyte_volumes:
        print(f"\n=== Analyte volume: {vol} µL ===")
        
        for rep in range(1, n_replicates + 1):
            print(f"  Replicate {rep} → rack1 vial {vial_pos}")
            
            # --- Wash FIRST (reset system) ---
            rsg.wash_sequence()
            
            # --- Rebuild airgap1 ---
            if vol == analyte_volumes[0] and rep == 1:
                print(f"Cold start: taking full airgap1 ({airgap1_volume} µL)")
                rsg.take_air_gap(airgap1_volume, rate=0.05)
            else:
                rsg.take_air_gap(half_airgap1, rate=0.05)
            
            # --- Prepickup ---
            rsg.prepickup(volume=prepickup_volume, rate=withdraw_rate)
            
            # --- Airgap2 ---
            rsg.take_air_gap(airgap2_volume, rate=0.05)
            
            # --- Pickup analyte ---
            rsg.pickup_from_vial(
                module_name="rack1",
                vial_pos=1,
                volume=vol,
                rate=withdraw_rate
            )
            
            # --- NEW: post-withdrawal equilibration ---
            #print(f"  Holding in stock for {withdraw_delay} s")
            #time.sleep(withdraw_delay)
            
            # --- Dispense ---
            dispense_volume = vol + prepickup_volume + airgap2_volume + half_airgap1
            rsg.dispense_in_vial(
                module_name="rack1",
                vial_pos=vial_pos,
                volume=dispense_volume,
                rate=infuse_rate
            )
            
            vial_pos += 1
            time.sleep(1)

    print("\n~~~ Syringe pump validation run complete ~~~")



In [16]:
def run_validation(
    rsg,
    analyte_volumes,
    start_vial_pos=2,
    n_replicates=2,
    prepickup_volume=20,   # µL
    airgap2_volume=5,      # µL - separates prepickup and analyte
    withdraw_rate=0.2,     # mL/min
    infuse_rate=0.5        # mL/min
):
    """
    Automated syringe pump validation workflow (no airgap1, no wash).
    Prepickup is directly added to working fluid.
    """
    
    print("=== Starting syringe pump validation run (no airgap1, no wash) ===")
    
    vial_pos = start_vial_pos

    for vol in analyte_volumes:
        print(f"\n=== Analyte volume: {vol} µL ===")
        
        for rep in range(1, n_replicates + 1):
            print(f"  Replicate {rep} → rack1 vial {vial_pos}")
            
            # --- Prepickup from wash vial (now directly into working fluid) ---
            rsg.prepickup(volume=prepickup_volume, rate=withdraw_rate)
            
            # --- Small airgap2 before analyte ---
            rsg.take_air_gap(airgap2_volume, rate=0.05)
            
            # --- Pickup analyte ---
            rsg.pickup_from_vial(
                module_name="rack1",
                vial_pos=1,
                volume=vol,
                rate=withdraw_rate
            )
            
            # --- Dispense all into dilution vial ---
            dispense_volume = vol + prepickup_volume + airgap2_volume
            rsg.dispense_in_vial(
                module_name="rack1",
                vial_pos=vial_pos,
                volume=dispense_volume,
                rate=infuse_rate
            )
            
            vial_pos += 1
            
            time.sleep(1)

    print("\n~~~ Syringe pump validation run complete ~~~")

In [19]:
analyte_volumes = [50, 40, 30, 20, 10]  # µL
n_replicates = 3
prepickup_volume = 20            # µL
airgap1_volume = 20              # µL (BACK ON)
airgap2_volume = 5               # µL
withdraw_rate = 0.05             # mL/min
infuse_rate = 0.5                # mL/min
#withdraw_delay = 2.0             # seconds (NEW)

run_validation(
    rsg,
    analyte_volumes=analyte_volumes,
    start_vial_pos=2,
    n_replicates=n_replicates,
    prepickup_volume=prepickup_volume,
    airgap1_volume=airgap1_volume,   # re-enabled
    airgap2_volume=airgap2_volume,
    withdraw_rate=withdraw_rate,
    infuse_rate=infuse_rate,
    #withdraw_delay=withdraw_delay    # added
)

=== Starting syringe pump validation run ===

=== Analyte volume: 50 µL ===
  Replicate 1 → rack1 vial 2
[Wash] Starting wash sequence
[Wash] Air gap: 5.0 µL
[Air Gap] 5.0uL @ 0.05mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[Wash] Pickup solvent: 100.0 µL
[Pickup] 100.0uL from rack2 vial 1 @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Wash] Dispense to waste: 105.0 µL
[Waste] 105.0uL to waste @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Wash] Complete
Cold start: taking full airgap1 (20 µL)
[Air Gap] 20uL @ 0.05mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Pre-pickup] 20uL from rack2 vial 1 @ 0.05 mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=

In [20]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0), 20)

In [4]:
g.home()

All axes homed successfully and Z is within safe limits


In [26]:
g.send_command("Get Client Connections")

'Get Client Connections: 111, Error:  Command not found.  Command expression could not be created.  Command not executed.'

In [8]:
print(g.current_x)
print(g.current_y)

0.0
0.0


In [7]:
g.move_xy(0,0)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


'Moved to X=0, Y=0. Result: Move XY: 2, Success'

In [13]:
g.go_to_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(106.0))

In [11]:
g.

Z below safe limit (110.00 < 130.00) — raising first.
All axes homed successfully and Z is within safe limits


In [59]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [47]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [53]:
g.move_z(120)

'Moved Z to 120. Result: Move Z: 2, Success'

In [12]:
g.home()

All axes homed successfully and Z is within safe limits


In [65]:
rate = 0.25          # mL/min
volume = 750           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.25 MM
Sent: @00*VOL750
Sent: @00*DIRINF


In [66]:
pump.start()

Sent: @00*RUN 1


'00I'

In [63]:
pump.stop()

Sent: @00*STP


'00P'

In [58]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(201.0), np.float64(106.0), 20)

In [10]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [79]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [80]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [73]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [74]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [19]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(8.0))

In [6]:
g.move_z(60)

'Moved Z to 60. Result: No valid response received.'

In [75]:
rsg.pickup_from_vial("rack1", 1, 4)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL4
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [76]:
g.go_to_vial("rack1", 25)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(159.5295))

In [77]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [78]:
rsg.dispense_in_vial("rack1", 25, 29)

#Note - analyte vol + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL,
# this 10uL is reintroduced before the wash sequence)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL29
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP
